
## Step 1: Imports

In [ ]:
import io, os, json, hashlib, subprocess
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import wavfile
from google.colab import files

try:
    from numba import njit
except ImportError:
    def njit(f):
        return f

## Step 2: The 5D chaotic model (paper Eq. 14), RK4 integer-order case

The first equation follows the paper's placement of the memristor parameter:

\[
\dot x_1=a(x_2-x_1)+0.25x_3+g(\delta+\gamma\tanh(x_5))x_3.
\]

The remaining equations are \(\dot x_2=bx_1-x_1x_3+cx_2-x_4\), \(\dot x_3=dx_1x_2-ex_3\), \(\dot x_4=x_1\), and \(\dot x_5=x_3\). This notebook uses classical RK4, so it implements **alpha = 1 only**. The paper's ABC fractional solver for alpha = 0.99 is not implemented here. The audio/video processing below is a project extension; the paper itself demonstrates the AIE image method.

In [ ]:
@njit
def model(s, p):
    x1, x2, x3, x4, x5 = s[0], s[1], s[2], s[3], s[4]
    a, b, c, d, e, g, delta, gamma = p[0], p[1], p[2], p[3], p[4], p[5], p[6], p[7]
    out = np.empty(5)
    out[0] = a * (x2 - x1) + 0.25 * x3 + g * (delta + gamma * np.tanh(x5)) * x3
    out[1] = b * x1 - x1 * x3 + c * x2 - x4
    out[2] = d * x1 * x2 - e * x3
    out[3] = x1
    out[4] = x3
    return out

@njit
def rk4_step(s, p, h):
    k1 = model(s, p)
    k2 = model(s + 0.5 * h * k1, p)
    k3 = model(s + 0.5 * h * k2, p)
    k4 = model(s + h * k3, p)
    return s + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

@njit
def advance(s, p, h, n_steps):
    for _ in range(n_steps):
        s = rk4_step(s, p, h)
    return s

@njit
def generate_chunk(s, p, h, n_samples, skip):
    # continues the orbit from state s; returns the samples AND the new state
    out = np.empty((n_samples, 3))
    for i in range(n_samples):
        for _ in range(skip):
            s = rk4_step(s, p, h)
        out[i, 0] = s[0]
        out[i, 1] = s[1]
        out[i, 2] = s[2]
    return out, s

## Step 3: Model key and processing settings

Use the same model parameters, initial conditions and processing options for encryption and decryption. `MAX_SECONDS = None` processes the complete clip; `SCALE_WIDTH = None` preserves the decoded frame dimensions. Set these to a short duration / smaller width only for a quick preview. In that case, recovery is exact only for the processed segment and resolution. Full-resolution video may require substantial RAM.

In [ ]:
KEY = dict(
    a=78, b=-16, c=58, d=35, e=15, g=1, delta=1, gamma=0.5,   # paper values
    x0=[0.0, 0.0, 8.0, 6.0, 0.0],                             # initial conditions
)
H_STEP    = 0.001
TRANSIENT = 10000
SKIP      = 10

MAX_SECONDS = None   # None = process the complete video; set e.g. 5 for a quick preview
SCALE_WIDTH = None   # None = keep the original decoded width and height; set e.g. 320 for a preview

def key_to_arrays(key):
    p  = np.array([key['a'], key['b'], key['c'], key['d'],
                   key['e'], key['g'], key['delta'], key['gamma']], dtype=np.float64)
    x0 = np.array(key['x0'], dtype=np.float64)
    return x0, p

## Step 4: Keystreams and encryption/decryption
- **Frames:** the orbit state continues between frames; RGB values use \(x_1,x_2,x_3\), then optionally XOR with the key image.
- **Audio:** signed 16-bit PCM samples are viewed as unsigned 16-bit words and XORed with a chaotic keystream, then optionally XORed with key audio.
- XOR is symmetric, so the same operations with the same initial state and key files reverse the operation exactly.

**Important limitation:** this construction is XOR-only and does not include a diffusion stage. High entropy / low correlation do not prove cryptographic security; one-element differential NPCR/UACI will be low by design. Do not reuse the same deterministic chaotic mask across different files and do not describe this prototype as proven secure encryption.

In [ ]:
class ChaoticStream:
    """Chaotic orbit that can be read in consecutive pieces (one piece per frame)."""
    def __init__(self, key):
        x0, self.p = key_to_arrays(key)
        self.s = advance(x0.copy(), self.p, H_STEP, TRANSIENT)    # discard the transient once

    def take(self, n):
        orbit, self.s = generate_chunk(self.s, self.p, H_STEP, n, SKIP)
        if not np.all(np.isfinite(orbit)):
            raise ValueError("The chaotic system blew up. Check your key values.")
        return orbit                                              # (n, 3)

    def frame_keystream(self, H, W):
        orbit = self.take(H * W)
        return (np.floor(np.abs(orbit) * 1e8).astype(np.int64) % 256).astype(np.uint8).reshape(H, W, 3)

    def audio_keystream(self, n, channels):
        orbit = self.take(n)[:, :channels]
        return (np.floor(np.abs(orbit) * 1e8).astype(np.int64) % 65536).astype(np.uint16)

def xor_frames(frames, key, key_img=None):
    """Encrypts OR decrypts a list of RGB frames (same function, XOR is symmetric)."""
    stream = ChaoticStream(key)
    out = []
    for f in frames:
        H, W, _ = f.shape
        c = f ^ stream.frame_keystream(H, W)
        if key_img is not None:
            c = c ^ key_img
        out.append(c)
    return out

def as_2d(x):
    return x.reshape(-1, 1) if x.ndim == 1 else x

def prepare_key_audio(key_audio, n, ch):
    k = as_2d(key_audio)
    k = np.tile(k, (int(np.ceil(n / k.shape[0])), 1))[:n]
    k = k[:, :ch] if k.shape[1] >= ch else np.tile(k[:, :1], (1, ch))
    return np.ascontiguousarray(k).view(np.uint16)

def xor_audio(audio, key, key_audio=None):
    """Encrypts OR decrypts an int16 audio array. Uses its own fresh chaotic stream."""
    a2 = np.ascontiguousarray(as_2d(audio))
    n, ch = a2.shape
    c = a2.view(np.uint16) ^ ChaoticStream(key).audio_keystream(n, ch)
    if key_audio is not None:
        c = c ^ prepare_key_audio(key_audio, n, ch)
    return c.view(np.int16).reshape(audio.shape)

## Step 5: Video input / output helpers (ffmpeg)

In [ ]:
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.decode(errors='ignore')[-800:])
    return r

def first_frame_size(path):
    # decode one frame so rotation metadata (phone videos) is already applied
    r = run(['ffmpeg', '-v', 'error', '-i', path, '-frames:v', '1', '-f', 'image2pipe', '-vcodec', 'png', '-'])
    return Image.open(io.BytesIO(r.stdout)).size            # (W, H)

def video_fps(path):
    r = run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=r_frame_rate',
             '-of', 'json', path])
    return json.loads(r.stdout)['streams'][0]['r_frame_rate']            # e.g. "30/1"

def target_size(W, H, scale_width):
    if scale_width:
        nw = scale_width // 2 * 2
        nh = int(round(H * nw / W)) // 2 * 2
    else:
        nw, nh = W, H
    return nw, nh

def read_frames(path, nw, nh, scale, max_seconds=None):
    cmd = ['ffmpeg', '-v', 'error']
    if max_seconds: cmd += ['-t', str(max_seconds)]
    cmd += ['-i', path, '-an']
    if scale:
        vf = f'scale={nw}:{nh}:flags=lanczos'
        cmd += ['-vf', vf]
    # Without scaling, decode at the native frame dimensions; do not crop pixels.
    cmd += ['-f', 'rawvideo', '-pix_fmt', 'rgb24', '-']
    raw = run(cmd).stdout
    size = nw * nh * 3
    return [np.frombuffer(raw[i:i + size], np.uint8).reshape(nh, nw, 3).copy() for i in range(0, len(raw) - size + 1, size)]

def read_audio(path, max_seconds=None):
    tmp = 'tmp_audio.wav'
    cmd = ['ffmpeg', '-y', '-v', 'error']
    if max_seconds: cmd += ['-t', str(max_seconds)]
    try:
        run(cmd + ['-i', path, '-vn', '-acodec', 'pcm_s16le', tmp])
    except RuntimeError:
        return None, None
    sr, data = wavfile.read(tmp)
    return sr, (np.ascontiguousarray(data) if data.size else None)

def write_lossless_mkv(path, frames, fps, sr=None, audio=None):
    # FFV1 (lossless video) + PCM 16-bit audio
    H, W, _ = frames[0].shape
    cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{W}x{H}', '-framerate', fps, '-i', '-']
    if audio is not None:
        wavfile.write('tmp_enc_audio.wav', sr, audio)
        cmd += ['-i', 'tmp_enc_audio.wav']
    cmd += ['-c:v', 'ffv1', '-pix_fmt', 'bgr0']
    if audio is not None: cmd += ['-c:a', 'pcm_s16le']  # do not truncate video to audio duration
    cmd += [path]
    subprocess.run(cmd, input=b''.join(f.tobytes() for f in frames), check=True, capture_output=True)

def write_viewable_mp4(path, frames, fps, sr=None, audio=None):
    # lossy H.264 + AAC: ONLY for watching (never decrypt from this)
    H, W, _ = frames[0].shape
    cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{W}x{H}', '-framerate', fps, '-i', '-']
    if audio is not None:
        wavfile.write('tmp_view_audio.wav', sr, audio)
        cmd += ['-i', 'tmp_view_audio.wav', '-c:a', 'aac', '-shortest']
    cmd += ['-c:v', 'libx264', '-crf', '17', '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2', '-pix_fmt', 'yuv420p', path]
    subprocess.run(cmd, input=b''.join(f.tobytes() for f in frames), check=True, capture_output=True)

## Step 6: Upload plain video (mp4, mov, avi, mkv ...)

In [ ]:
uploaded = files.upload()
video_name = list(uploaded.keys())[0]
with open(video_name, 'wb') as f:
    f.write(uploaded[video_name])

size_mb = len(uploaded[video_name]) / 1e6
print(f"Uploaded {video_name}: {size_mb:.2f} MB")
if len(uploaded[video_name]) < 10_000:
    raise ValueError(f"{video_name} is only {len(uploaded[video_name])} bytes, so it is empty or a placeholder, not a real video. "
                     "Download the full file to your computer first (e.g. from Drive/OneDrive: 'Download', or make it 'available offline'), then upload it again.")
try:
    W0, H0 = first_frame_size(video_name)
except RuntimeError as err:
    raise ValueError(f"ffmpeg cannot read {video_name}. The file may be corrupted or incomplete (for example a recording that was not finished). "
                     "Try re-exporting it or converting it to mp4/H.264, then upload again.\n\nffmpeg said: " + str(err)) from None
NW, NH = target_size(W0, H0, SCALE_WIDTH)
FPS = video_fps(video_name)

plain_frames = read_frames(video_name, NW, NH, scale=bool(SCALE_WIDTH), max_seconds=MAX_SECONDS)
sr, plain_audio = read_audio(video_name, MAX_SECONDS)

print(f"Loaded {video_name}: original {W0}x{H0} -> working size {NW}x{NH}, {len(plain_frames)} frames at {FPS} fps")
print("Audio:", "none found" if plain_audio is None else f"{sr} Hz, shape {plain_audio.shape}")
print(f"Chaotic samples needed for frames: {len(plain_frames) * NW * NH:,}")
plt.imshow(plain_frames[0]); plt.title("Frame 1 (plain)"); plt.axis('off'); plt.show()

## Step 7 (optional): Key video
Like the key image in the paper: decryption needs the chaotic key **and** the same key file(s).  
`KEY_MODE = "video"` (default): upload **one key video**. Its **first frame** is the key image and its **audio track** is the key audio.  
`KEY_MODE = "separate"`: upload a key image and a key audio file separately. `KEY_MODE = "none"`: no extra key.  
For decryption you must provide the same key video again, without re-exporting or re-compressing it.


In [ ]:
KEY_MODE = "video"      # "video" = ONE key video (image + audio)  |  "separate" = key image and key audio files  |  "none"

def load_key_image(data, name, nw, nh):
    """Key image from an image file. If a VIDEO is given instead, its first frame is used (same every time)."""
    if len(data) < 1000:
        raise ValueError(f"{name} is only {len(data)} bytes: empty or a placeholder. Download the real file and upload it again.")
    try:
        im = Image.open(io.BytesIO(data)).convert('RGB')
    except Exception:
        with open(name, 'wb') as f:
            f.write(data)
        try:
            r = run(['ffmpeg', '-v', 'error', '-i', name, '-frames:v', '1', '-f', 'image2pipe', '-vcodec', 'png', '-'])
            im = Image.open(io.BytesIO(r.stdout)).convert('RGB')
        except Exception:
            raise ValueError(f"{name} is neither a readable image nor a video. Upload a png/jpg (or a valid video).") from None
    return np.array(im.resize((nw, nh)), dtype=np.uint8)

def load_key_audio(data, name):
    """Key audio as 16-bit PCM. Any audio/video file is converted with ffmpeg if it is not already a 16-bit WAV."""
    if len(data) < 1000:
        raise ValueError(f"{name} is only {len(data)} bytes: empty or a placeholder. Download the real file and upload it again.")
    try:
        _, k = wavfile.read(io.BytesIO(data))
        if k.dtype == np.int16:
            return np.ascontiguousarray(k)
    except Exception:
        pass
    with open(name, 'wb') as f:
        f.write(data)
    try:
        run(['ffmpeg', '-y', '-v', 'error', '-i', name, '-vn', '-acodec', 'pcm_s16le', 'tmp_key_audio.wav'])
        _, k = wavfile.read('tmp_key_audio.wav')
    except Exception:
        raise ValueError(f"{name} has no readable audio. Upload a 16-bit PCM WAV (or a file with an audio track).") from None
    return np.ascontiguousarray(k)

key_img = None
key_audio = None

if KEY_MODE == "video":
    # ONE key video: its first frame = key image, its audio track = key audio
    print("Upload the KEY VIDEO (use a different video from the plain one):")
    up = files.upload()
    kname = list(up.keys())[0]
    key_img = load_key_image(up[kname], kname, NW, NH)
    print(f"Key video: {kname} -> key image = its first frame")
    if plain_audio is not None:
        try:
            key_audio = load_key_audio(up[kname], kname)
            print(f"Key audio = its audio track ({key_audio.shape[0]} samples)")
        except ValueError:
            print("The key video has no audio track: audio will be encrypted with the chaotic key only.")

elif KEY_MODE == "separate":
    print("Upload the KEY IMAGE (png/jpg):")
    up = files.upload()
    kname = list(up.keys())[0]
    key_img = load_key_image(up[kname], kname, NW, NH)
    print("Key image:", kname)
    if plain_audio is not None:
        print("Upload the KEY AUDIO (16-bit PCM wav):")
        up = files.upload()
        aname = list(up.keys())[0]
        key_audio = load_key_audio(up[aname], aname)
        print("Key audio:", aname)
else:
    print("No key image / key audio used (chaotic key only).")

## Step 8: Encrypt frames and audio, save the lossless encrypted video

In [ ]:
enc_frames = xor_frames(plain_frames, KEY, key_img)
enc_audio  = xor_audio(plain_audio, KEY, key_audio) if plain_audio is not None else None

write_lossless_mkv("encrypted_video.mkv", enc_frames, FPS, sr, enc_audio)          # decrypt from THIS file
write_viewable_mp4("encrypted_preview.mp4", enc_frames, FPS, sr, enc_audio)        # only to look at it
files.download("encrypted_video.mkv")
files.download("encrypted_preview.mp4")

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for a, i in zip(ax, [0, len(plain_frames)//2, len(plain_frames)-1]):
    a.imshow(enc_frames[i]); a.set_title(f"Encrypted frame {i+1}"); a.axis('off')
plt.show()

## Step 9: Decrypt
Reads the lossless `encrypted_video.mkv`, decrypts every frame and the audio, and rebuilds a normal mp4.  
`LOAD_ENCRYPTED_FROM_FILE = True` uploads a saved `encrypted_video.mkv` instead (run Steps 3 and 7 first with the same key and key files).

In [ ]:
LOAD_ENCRYPTED_FROM_FILE = False

enc_path = "encrypted_video.mkv"
if LOAD_ENCRYPTED_FROM_FILE:
    up = files.upload()
    enc_path = list(up.keys())[0]
    with open(enc_path, 'wb') as f:
        f.write(up[enc_path])

cipher_frames = read_frames(enc_path, NW, NH, scale=False)
cipher_sr, cipher_audio = read_audio(enc_path)

dec_frames = xor_frames(cipher_frames, KEY, key_img)
dec_audio  = xor_audio(cipher_audio, KEY, key_audio) if cipher_audio is not None else None

frames_ok = len(dec_frames) == len(plain_frames) and all(np.array_equal(a, b) for a, b in zip(dec_frames, plain_frames))
audio_ok  = (dec_audio is None and plain_audio is None) or (dec_audio is not None and plain_audio is not None
             and dec_audio.shape == plain_audio.shape and np.array_equal(dec_audio, plain_audio))
print("All frames identical to the original:", frames_ok, f"({len(dec_frames)} frames)")
print("Audio identical to the original     :", audio_ok)

write_viewable_mp4("decrypted_video.mp4", dec_frames, FPS, cipher_sr, dec_audio)
files.download("decrypted_video.mp4")

## Step 10: Diagnostic checks

These checks show entropy, spatial/temporal correlation, exact recovery, and wrong-key behavior on the processed frames/audio. They do not prove cryptographic security. For differential NPCR/UACI, use the separate 12-test notebook: this XOR-only design has no diffusion, so one changed plaintext element changes one ciphertext element. The lossless MKV is the file to decrypt; the H.264/AAC MP4 is only a viewing preview and is lossy.

In [ ]:
def entropy(ch):
    h = np.bincount(ch.ravel(), minlength=256) / ch.size
    h = h[h > 0]
    return -np.sum(h * np.log2(h))

def corr_pair(a, b):
    return np.corrcoef(a.ravel().astype(float), b.ravel().astype(float))[0, 1]

def spatial_corr(img, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    r = rng.integers(0, img.shape[0], n); c = rng.integers(0, img.shape[1] - 1, n)
    return np.corrcoef(img[r, c, 0].astype(float), img[r, c + 1, 0].astype(float))[0, 1]

def mse(a, b):  return float(np.mean((a.astype(float) - b.astype(float)) ** 2))
def psnr(a, b):
    m = mse(a, b)
    return float('inf') if m == 0 else 10 * np.log10(255 ** 2 / m)
def npcr_uaci(a, b):
    return (a != b).mean() * 100, (np.abs(a.astype(float) - b.astype(float)) / 255).mean() * 100

# 1) Entropy and correlation on a few frames
idx = [0, len(plain_frames) // 2, len(plain_frames) - 1]
for i in idx:
    print(f"Frame {i+1:>4}  entropy plain {[round(float(entropy(plain_frames[i][..., k])), 3) for k in range(3)]}"
          f"  enc {[round(float(entropy(enc_frames[i][..., k])), 3) for k in range(3)]}"
          f"  | spatial corr plain {spatial_corr(plain_frames[i]):.3f} enc {spatial_corr(enc_frames[i]):.3f}")

# 2) Temporal correlation: consecutive frames (very high in a normal video, ~0 after encryption)
if len(plain_frames) > 1:
    tp = np.mean([corr_pair(plain_frames[i], plain_frames[i + 1]) for i in range(min(20, len(plain_frames) - 1))])
    te = np.mean([corr_pair(enc_frames[i],   enc_frames[i + 1])   for i in range(min(20, len(plain_frames) - 1))])
    print(f"Correlation between consecutive frames   plain: {tp:.4f}   encrypted: {te:.4f}")

# 3) Histogram of frame 1
plt.figure(figsize=(10, 3.2))
plt.subplot(1, 2, 1); plt.hist(plain_frames[0].ravel(), bins=256, range=(0, 255)); plt.title("Histogram: plain frame 1")
plt.subplot(1, 2, 2); plt.hist(enc_frames[0].ravel(),   bins=256, range=(0, 255)); plt.title("Histogram: encrypted frame 1")
plt.tight_layout(); plt.show()

# 4) Frame 1: correct key, wrong chaotic key, wrong key image
f_enc = cipher_frames[0]
ok = xor_frames([f_enc], KEY, key_img)[0]
bad_key = dict(KEY); bad_key['x0'] = list(KEY['x0']); bad_key['x0'][0] += 1e-10
wr = xor_frames([f_enc], bad_key, key_img)[0]
n, u = npcr_uaci(ok, plain_frames[0]);  print(f"Frame 1 correct key       : MSE = {mse(ok, plain_frames[0]):.5f}  PSNR = {psnr(ok, plain_frames[0])}  NPCR = {n:.4f} %  UACI = {u:.4f} %")
n, u = npcr_uaci(wr, plain_frames[0]);  print(f"Frame 1 wrong chaotic key : MSE = {mse(wr, plain_frames[0]):.1f}  PSNR = {psnr(wr, plain_frames[0]):.2f} dB  NPCR = {n:.2f} %  UACI = {u:.2f} %")
fig, ax = plt.subplots(1, 3 if key_img is not None else 2, figsize=(11, 3.5))
ax[0].imshow(ok); ax[0].set_title("Correct key"); ax[1].imshow(wr); ax[1].set_title("Key off by 1e-10")
if key_img is not None:
    wk = xor_frames([f_enc], KEY, 255 - key_img)[0]
    n, u = npcr_uaci(wk, plain_frames[0]); print(f"Frame 1 wrong key image   : MSE = {mse(wk, plain_frames[0]):.1f}  PSNR = {psnr(wk, plain_frames[0]):.2f} dB  NPCR = {n:.2f} %  UACI = {u:.2f} %")
    ax[2].imshow(wk); ax[2].set_title("Wrong key image")
for a in ax: a.axis('off')
plt.show()

# 5) Audio checks
if plain_audio is not None:
    def snr_db(ref, test):
        noise = np.sum((ref.astype(float) - test.astype(float)) ** 2)
        return float('inf') if noise == 0 else 10 * np.log10(np.sum(ref.astype(float) ** 2) / noise)
    v = as_2d(enc_audio).view(np.uint8).ravel()
    h = np.bincount(v, minlength=256) / v.size; h = h[h > 0]
    print(f"Audio byte entropy (max 8): plain {entropy(as_2d(plain_audio).view(np.uint8)):.4f}  encrypted {-np.sum(h * np.log2(h)):.4f}")
    bad = xor_audio(cipher_audio, bad_key, key_audio)
    print(f"Audio wrong chaotic key: SNR = {snr_db(plain_audio, bad):.2f} dB (negative = decryption failed)")

# 6) Keystream fingerprint (same key => same value in every session)
chk = ChaoticStream(KEY).take(1000)
print("Keystream fingerprint:", hashlib.sha256(chk.tobytes()).hexdigest()[:16])